# 05. SAML Vulnerabilities

## Background

Security Assertion Markup Language (SAML) is the XML-based single sign-on protocol that predates OIDC and still powers enterprise SSO integrations with legacy systems. Active Directory Federation Services (ADFS), Okta's enterprise connectors, and most SAML 2.0 IdPs handle authentication for millions of corporate users.

SAML's XML foundation creates vulnerability surfaces that JSON-based JWT protocols avoid entirely. XML signature wrapping, XML comment injection, and SAML replay attacks have been used in real-world attacks against Google Workspace, GitHub Enterprise, and various enterprise applications.

## What You'll Learn

- SAML assertion structure: NameID, Attributes, AuthnStatement, Conditions
- XML Signature Wrapping (XSW): the most impactful SAML attack class
- Comment injection: splitting NameID with XML comments to bypass username checks
- Replay attack: reusing a valid assertion outside its validity window
- Secure SAML parsing: validating signature position, not just presence

## Why This Matters

SAML XSW vulnerabilities have been found in Duo Security (2018), GitHub Enterprise, OneLogin, and the SAML library used by the US Department of Defense. The root cause is always the same: parsers that verify the signature exists but not that it covers the assertion actually used for authorization.

In [ ]:
import xml.etree.ElementTree as ET
import hashlib, base64, time, re
from dataclasses import dataclass

print('SAML vulnerability demos loaded')


## 1. SAML Assertion Structure

In [ ]:
SAML_ASSERTION = '''<?xml version="1.0" encoding="UTF-8"?>
<samlp:Response xmlns:samlp="urn:oasis:names:tc:SAML:2.0:protocol"
                ID="_response_001" Version="2.0">
  <saml:Assertion xmlns:saml="urn:oasis:names:tc:SAML:2.0:assertion"
                  ID="_assertion_001">
    <saml:Issuer>https://idp.example.com</saml:Issuer>
    <ds:Signature xmlns:ds="http://www.w3.org/2000/09/xmldsig#">
      <ds:SignedInfo>
        <ds:Reference URI="#_assertion_001"/>
      </ds:SignedInfo>
      <ds:SignatureValue>BASE64_SIGNATURE_HERE</ds:SignatureValue>
    </ds:Signature>
    <saml:Conditions NotBefore="2024-01-01T00:00:00Z"
                     NotOnOrAfter="2024-01-01T00:05:00Z">
      <saml:AudienceRestriction>
        <saml:Audience>https://sp.example.com</saml:Audience>
      </saml:AudienceRestriction>
    </saml:Conditions>
    <saml:AuthnStatement AuthnInstant="2024-01-01T00:00:00Z"/>
    <saml:AttributeStatement>
      <saml:Attribute Name="email">
        <saml:AttributeValue>alice@example.com</saml:AttributeValue>
      </saml:Attribute>
      <saml:Attribute Name="role">
        <saml:AttributeValue>user</saml:AttributeValue>
      </saml:Attribute>
    </saml:AttributeStatement>
  </saml:Assertion>
</samlp:Response>'''

tree = ET.fromstring(SAML_ASSERTION)
ns = {
    'saml':  'urn:oasis:names:tc:SAML:2.0:assertion',
    'samlp': 'urn:oasis:names:tc:SAML:2.0:protocol',
    'ds':    'http://www.w3.org/2000/09/xmldsig#',
}
email = tree.find('.//saml:AttributeValue[..[@Name="email"]]', ns)
print(f'Parsed email: {email.text if email is not None else "not found"}')


## 2. XML Signature Wrapping (XSW)

In XSW, the attacker duplicates the assertion: one copy is signed (valid signature), another is unsigned but placed where the SP's parser reads it for authorization. If the SP validates 'is there a signature?' rather than 'does the signature cover this specific assertion?', the attack succeeds.

In [ ]:
XSW_ATTACK = '''<?xml version="1.0" encoding="UTF-8"?>
<samlp:Response xmlns:samlp="urn:oasis:names:tc:SAML:2.0:protocol" ID="_response_001">
  <saml:Assertion xmlns:saml="urn:oasis:names:tc:SAML:2.0:assertion" ID="_assertion_EVIL">
    <!-- THIS is what the vulnerable SP reads for authz -->
    <saml:Issuer>https://idp.example.com</saml:Issuer>
    <saml:AttributeStatement>
      <saml:Attribute Name="email">
        <saml:AttributeValue>alice@example.com</saml:AttributeValue>
      </saml:Attribute>
      <saml:Attribute Name="role">
        <saml:AttributeValue>admin</saml:AttributeValue>
      </saml:Attribute>
    </saml:AttributeStatement>
  </saml:Assertion>
  <ds:Signature xmlns:ds="http://www.w3.org/2000/09/xmldsig#">
    <ds:SignedInfo>
      <ds:Reference URI="#_assertion_LEGIT"/>
    </ds:SignedInfo>
    <ds:SignatureValue>VALID_SIGNATURE_OVER_LEGIT_ASSERTION</ds:SignatureValue>
    <!-- Signed assertion wrapped inside signature (ignored by some parsers) -->
    <saml:Assertion xmlns:saml="urn:oasis:names:tc:SAML:2.0:assertion" ID="_assertion_LEGIT">
      <saml:AttributeStatement>
        <saml:Attribute Name="role">
          <saml:AttributeValue>user</saml:AttributeValue>
        </saml:Attribute>
      </saml:AttributeStatement>
    </saml:Assertion>
  </ds:Signature>
</samlp:Response>'''

def parse_vulnerable(saml_xml):
    tree = ET.fromstring(saml_xml)
    ns = {'saml':'urn:oasis:names:tc:SAML:2.0:assertion',
          'ds':'http://www.w3.org/2000/09/xmldsig#'}
    sig = tree.find('.//ds:Signature', ns)
    if sig is None: return None, 'no signature'
    # BUG: only checks signature exists, not WHICH assertion it covers
    assertion = tree.find('saml:Assertion', ns)   # gets FIRST = evil one
    role_el = assertion.find('.//saml:AttributeValue[..[@Name="role"]]', ns) if assertion is not None else None
    role = role_el.text if role_el is not None else 'unknown'
    return role, None

def parse_secure(saml_xml):
    tree = ET.fromstring(saml_xml)
    ns = {'saml':'urn:oasis:names:tc:SAML:2.0:assertion',
          'ds':'http://www.w3.org/2000/09/xmldsig#'}
    sig = tree.find('.//ds:Signature', ns)
    if sig is None: return None, 'no signature'
    # Secure: find the assertion the signature actually references
    ref = sig.find('ds:SignedInfo/ds:Reference', ns)
    ref_id = ref.get('URI','').lstrip('#') if ref is not None else None
    # Find assertion with that ID — must be direct child of Response, not inside Signature
    assertion = None
    for a in tree.findall('saml:Assertion', ns):
        if a.get('ID') == ref_id:
            assertion = a; break
    if assertion is None: return None, f'Signed assertion {ref_id} not found at top level'
    role_el = assertion.find('.//saml:AttributeValue[..[@Name="role"]]', ns)
    role = role_el.text if role_el is not None else 'unknown'
    return role, None

role_vuln, _ = parse_vulnerable(XSW_ATTACK)
role_safe, err_safe = parse_secure(XSW_ATTACK)
print(f'Vulnerable parser role: {role_vuln}  <- attacker escalated to admin!')
print(f'Secure parser role:     {role_safe}  error={err_safe}')


## 3. Comment Injection Attack

In [ ]:
# XML comment injection: split a username string with a comment
# The signature covers 'alice@example.com' but the parser strips comments
# and reads 'alice@example.com<!--admin-->\nadmin@example.com' as 'admin@example.com'

def extract_username_vulnerable(name_id_text):
    # Vulnerable: strips comments before comparison (allows injection)
    cleaned = re.sub(r'<!--.*?-->', '', name_id_text, flags=re.DOTALL)
    return cleaned.strip()

def extract_username_secure(raw_xml_text):
    # Secure: reject any XML comment in NameID
    if '<!--' in raw_xml_text:
        raise ValueError('XML comments not allowed in NameID')
    return raw_xml_text.strip()

injected_nameid = 'alice@example.com<!--unused-->\nadmin@example.com'
print(f'Injected NameID: {repr(injected_nameid)}')
print(f'Vulnerable parse: {extract_username_vulnerable(injected_nameid)}')
try:
    print(f'Secure parse:     {extract_username_secure(injected_nameid)}')
except ValueError as e:
    print(f'Secure parse:     REJECTED — {e}')
